# 00 Suite Executive Rollup

**Customer360 Navigator Enterprise Suite -- suite-wide capstone report.**

This notebook is a **new, additive** deliverable. It is **not** part of the original Master Plan's 8-BP gate sequence -- it was requested directly by the project owner as a suite-wide capstone that aggregates:

- BP1 through BP7's own **already-produced Gate 7 executive rollups** (`executive_rollup_manifest.json`, plus BP1's/BP2's own Gate 6 governance block for the one disclosed tier-derivation case), and
- BP8's own **Gate 1 / Gate 2 / Gate 3** status (BP8 is a Gold-table / Power BI aggregation layer, not a modeling BP, so it has no production-recommendation-tier concept).

into one consolidated dashboard (HTML), report (DOCX), workbook (XLSX), and deck (PPTX).

**This notebook is strictly READ-ONLY over every BP1-BP8 file it opens.** It never writes to, modifies, or re-derives a metric already computed by any BP's own Gate 7 (or BP8's own Gate 1/2/3). The only place it writes is under three brand-new locations: this notebook's own `artifacts/` folder, `reports/00_suite_executive_rollup/`, and (as the deliverable itself) `src/reporting/suite_rollup_helpers.py` / `src/reporting/templates/00_suite_dashboard_template.html`, which are not touched by the notebook at runtime.

A before/after file-fingerprint check (Sections 4 and 10) structurally proves that no BP1-BP8 file was mutated by this run.

This notebook contains no financial-impact section and no assumption-based content, consistent with this project's project-wide ban on both.

## 1. Project-root resolution

In [ ]:
import os
from pathlib import Path

# Environment-robustness fallback (added 2026-09-25, additive only): on some
# Windows Jupyter launch setups the kernel's cwd does not match this
# notebook's own folder, which breaks the upward-walk resolver below. This
# project's own PROJECT_STRUCTURE_LOCKED.md fixes the project at one single,
# known real path on this machine (never written anywhere else, a standing
# project rule) - so pre-populating C360_PROJECT_ROOT with that real path
# here, ONLY if it isn't already set, is a safe disclosed default rather
# than a silent guess. The resolver's own precedence is unchanged: an env
# var the user has already set (e.g. for a different checkout) still wins,
# and the resolver still validates configs/+src/ exist before trusting it.
os.environ.setdefault(
    "C360_PROJECT_ROOT", r"C:\Users\rnand\Documents\Customer360_Navigator_Enterprise_Suite"
)

def _resolve_project_root(start_path: Path, max_levels: int = 6) -> Path:
    """Resolve the Customer360 project root: env override first, then a
    bounded upward walk looking for a directory containing both configs/
    and src/. This is the identical pattern used by every notebook in
    this project -- never a hardcoded absolute path."""
    env_override = os.environ.get("C360_PROJECT_ROOT")
    if env_override:
        candidate = Path(env_override).expanduser().resolve()
        if (candidate / "configs").is_dir() and (candidate / "src").is_dir():
            return candidate
        raise RuntimeError(
            f"C360_PROJECT_ROOT={env_override!r} does not contain both "
            "configs/ and src/ subdirectories."
        )

    current = Path(start_path).expanduser().resolve()
    for _ in range(max_levels + 1):
        if (current / "configs").is_dir() and (current / "src").is_dir():
            return current
        if current.parent == current:
            break
        current = current.parent

    raise RuntimeError(
        f"Could not resolve the Customer360 project root by walking up "
        f"from {start_path!r} (checked up to {max_levels} levels, looking "
        "for a directory containing both configs/ and src/). Set "
        "C360_PROJECT_ROOT explicitly to override."
    )


# This notebook lives at notebooks/00_suite_executive_rollup/, mirroring
# the one existing top-level, non-BP-specific notebook precedent
# (notebooks/00_hardware_benchmark/00_hardware_benchmark.ipynb).
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = _resolve_project_root(NOTEBOOK_DIR)
print(f"Resolved PROJECT_ROOT = {PROJECT_ROOT}")

## 2. Imports

In [ ]:
# === WARP: IMPORTS ===
# noqa: E402 below -- these are top-of-CELL imports (this is cell 2 of
# the notebook); flake8 only flags them as "not at top of file" when a
# notebook's cells are concatenated into one flat .py for linting.
import json  # noqa: E402
import sys  # noqa: E402
from datetime import datetime, timezone  # noqa: E402

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from reporting import suite_rollup_helpers as srh  # noqa: E402
# === END WARP: IMPORTS ===

## 3. Before-read fingerprints of every BP1-BP8 file this notebook opens

Mirrors BP8 Gate2/Gate3's own established defense-in-depth pattern: fingerprint every source file before touching it, and re-fingerprint at the end (Section 8) to structurally prove no mutation occurred.

In [ ]:
SOURCE_PATHS = []

# BP1-BP7 own executive rollup manifests
for _bp in srh.BP_ORDER:
    SOURCE_PATHS.append(
        PROJECT_ROOT / "notebooks" / srh.BP_FOLDER_NAMES[_bp] / "artifacts"
        / "executive_rollup_manifest.json"
    )

# BP1/BP2 configs (gate6_governance block, for the tier-derivation rule)
SOURCE_PATHS.append(PROJECT_ROOT / "configs" / "bp1_customer_intent_classification.yaml")
SOURCE_PATHS.append(PROJECT_ROOT / "configs" / "bp2_customer_friction_classification.yaml")

# BP1/BP2 own rendered dashboards -- READ ONLY, for the disclosed
# cross-check assertion. No other BP\'s .html/.docx/.xlsx/.pptx is ever
# opened by this notebook.
BP1_DASHBOARD_HTML = (
    PROJECT_ROOT / "reports" / "bp1_customer_intent_classification"
    / "executive_rollup" / "bp1_executive_rollup_dashboard.html"
)
BP2_DASHBOARD_HTML = (
    PROJECT_ROOT / "reports" / "bp2_customer_friction_classification"
    / "executive_rollup" / "bp2_executive_rollup_dashboard.html"
)
SOURCE_PATHS += [BP1_DASHBOARD_HTML, BP2_DASHBOARD_HTML]

# BP8 own Gate1/2/3 artifacts + config
BP8_ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / srh.BP8_FOLDER_NAME / "artifacts"
SOURCE_PATHS += [
    BP8_ARTIFACTS_DIR / "policy.json",
    BP8_ARTIFACTS_DIR / "gate2_gold_table_manifest.json",
    BP8_ARTIFACTS_DIR / "gate3_decision_engine_kpi_manifest.json",  # may not exist yet
    PROJECT_ROOT / "configs" / "bp8_executive_product_analytics.yaml",
]

FINGERPRINTS_BEFORE = srh.fingerprint_paths(SOURCE_PATHS)
print(f"Fingerprinted {len(FINGERPRINTS_BEFORE)} source files before read.")
GATE3_PATH = BP8_ARTIFACTS_DIR / "gate3_decision_engine_kpi_manifest.json"
print(f"BP8 Gate 3 manifest exists at read time: {GATE3_PATH.exists()}")

## 4. Load BP1-BP7 summaries + BP8 Gate 1/2/3 status

In [ ]:
BUNDLE = srh.load_all_bp_summaries(PROJECT_ROOT)

# Cross-check assertion (defense-in-depth, never the primary source):
# the derived BP1/BP2 tier must literally appear in each BP\'s own
# rendered dashboard HTML.
srh.cross_check_tier_in_html(BP1_DASHBOARD_HTML, BUNDLE["bp1"].tier, "BP1")
srh.cross_check_tier_in_html(BP2_DASHBOARD_HTML, BUNDLE["bp2"].tier, "BP2")
print("BP1/BP2 tier cross-check against rendered dashboard HTML: PASSED")

for _bp in srh.BP_ORDER:
    _r = BUNDLE[_bp]
    print(f"{_bp.upper():4s} champion={_r.champion!r:35s} "
          f"tier=({_r.tier_source}) {_r.tier!r}")
print("BP8 gate1_confirmed:", BUNDLE["bp8"]["gate1_confirmed"],
      "| gate2_confirmed:", BUNDLE["bp8"]["gate2_confirmed"],
      "| gate3_status:", BUNDLE["bp8"]["gate3_status"])

## 5. Suite-wide KPIs

In [ ]:
KPIS = srh.compute_suite_kpis(BUNDLE)
KPIS

## 6. Per-BP status table + summary charts

In [ ]:
STATUS_DF = srh.build_status_dataframe(BUNDLE)
STATUS_DF

In [ ]:
TIER_FIG = srh.fig_tier_distribution(STATUS_DF)
TIER_PNG_BYTES = srh.fig_to_png_bytes(TIER_FIG)
TIER_PNG_B64 = srh.png_bytes_to_base64(TIER_PNG_BYTES)

PYTEST_FIG = srh.fig_pytest_passed_per_bp(BUNDLE)
PYTEST_PNG_BYTES = srh.fig_to_png_bytes(PYTEST_FIG)
PYTEST_PNG_B64 = srh.png_bytes_to_base64(PYTEST_PNG_BYTES)

print("Rendered tier-distribution and pytest-passed charts.")

## 7. Render + write dashboard HTML, DOCX report, XLSX workbook, PPTX deck

All four outputs are written under the brand-new `reports/00_suite_executive_rollup/` folder only.

In [ ]:
REPORTS_DIR = PROJECT_ROOT / "reports" / "00_suite_executive_rollup"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

DASHBOARD_HTML_PATH = REPORTS_DIR / "00_suite_executive_rollup_dashboard.html"
DOCX_PATH = REPORTS_DIR / "00_suite_executive_rollup_report.docx"
XLSX_PATH = REPORTS_DIR / "00_suite_executive_rollup_workbook.xlsx"
PPTX_PATH = REPORTS_DIR / "00_suite_executive_rollup_deck.pptx"
PDF_PATH = REPORTS_DIR / "00_suite_executive_rollup_report.pdf"

TEMPLATE_PATH = SRC_PATH / "reporting" / "templates" / "00_suite_dashboard_template.html"

dashboard_html = srh.render_dashboard_html(
    TEMPLATE_PATH, BUNDLE, KPIS, STATUS_DF, TIER_PNG_B64, PYTEST_PNG_B64,
)
DASHBOARD_HTML_PATH.write_text(dashboard_html, encoding="utf-8")

srh.write_docx_report(DOCX_PATH, BUNDLE, KPIS, STATUS_DF)
srh.write_xlsx_workbook(XLSX_PATH, BUNDLE, KPIS, STATUS_DF)
srh.write_pptx_deck(PPTX_PATH, BUNDLE, KPIS, STATUS_DF, TIER_PNG_BYTES)

# PDF is converted FROM the DOCX just written above (never a second,
# independently-authored document -- see write_pdf_report's docstring).
# Kept OUT of OUTPUT_PATHS / the hard-required-files check in Section 9:
# PDF conversion depends on Microsoft Word or LibreOffice being installed,
# which this deliverable is not authorized to assume -- Section 9's own
# check (h) treats a missing PDF as a soft WARN, never a hard FAIL.
PDF_RESULT = srh.write_pdf_report(DOCX_PATH, PDF_PATH)
print(f"report_pdf: {PDF_RESULT['status']} "
      f"({PDF_RESULT.get('method', PDF_RESULT.get('reason', ''))})")

OUTPUT_PATHS = {
    "dashboard_html": str(DASHBOARD_HTML_PATH),
    "report_docx": str(DOCX_PATH),
    "workbook_xlsx": str(XLSX_PATH),
    "deck_pptx": str(PPTX_PATH),
}
for _name, _path in OUTPUT_PATHS.items():
    _size = Path(_path).stat().st_size
    print(f"{_name}: {_path} ({_size:,} bytes)")


## 8. After-read fingerprints + suite manifest

Re-fingerprint every source file this notebook read, then assemble and write this notebook's own manifest sidecar.

In [ ]:
FINGERPRINTS_AFTER = srh.fingerprint_paths(SOURCE_PATHS)

GENERATED_AT_UTC = datetime.now(timezone.utc).isoformat()

MANIFEST = srh.suite_rollup_manifest(
    bundle=BUNDLE,
    kpis=KPIS,
    fingerprints_before=FINGERPRINTS_BEFORE,
    fingerprints_after=FINGERPRINTS_AFTER,
    output_paths=OUTPUT_PATHS,
    generated_at_utc=GENERATED_AT_UTC,
    pdf_result=PDF_RESULT,
)

ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "00_suite_executive_rollup" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = ARTIFACTS_DIR / "suite_executive_rollup_manifest.json"
MANIFEST_PATH.write_text(json.dumps(MANIFEST, indent=2), encoding="utf-8")
print(f"Wrote suite manifest to {MANIFEST_PATH} "
      f"({MANIFEST_PATH.stat().st_size:,} bytes)")


## 9. Structural integrity checks

In [ ]:
# (a) zero mutation of every BP1-BP8 file this notebook read
srh.assert_no_mutation(FINGERPRINTS_BEFORE, FINGERPRINTS_AFTER)
print("(a) PASSED: no BP1-BP8 source file was mutated by this run.")

# (b) structural fact about the suite
assert KPIS["n_bps_total"] == 8, "n_bps_total must be 8 (Master Plan structural fact)"
print("(b) PASSED: n_bps_total == 8")

# (c) BP1/BP2 tier cross-check (already asserted in Section 4; re-assert here
#     as part of the consolidated integrity-check cell)
srh.cross_check_tier_in_html(BP1_DASHBOARD_HTML, BUNDLE["bp1"].tier, "BP1")
srh.cross_check_tier_in_html(BP2_DASHBOARD_HTML, BUNDLE["bp2"].tier, "BP2")
print("(c) PASSED: BP1/BP2 derived tier cross-checked against rendered dashboard HTML")

# (d) financial-impact / assumption-based-content ban, for every BP and for
#     the suite\'s own manifest
srh.assert_no_financial_or_assumption_content(BUNDLE)
assert MANIFEST["contains_financial_impact_section"] is False
assert MANIFEST["contains_assumption_based_content"] is False
print("(d) PASSED: no financial-impact or assumption-based content anywhere in scope")

# (e) the 4 real output files were written and are non-empty
for _name, _path in OUTPUT_PATHS.items():
    _p = Path(_path)
    assert _p.exists() and _p.stat().st_size > 0, f"{_name} missing or empty: {_p}"
print("(e) PASSED: all 4 output files exist and are non-empty")

# (f) BP8 Gate-3-manifest-absent case does not crash -- assert the code path
#     taken matches Path.exists(), not a fixed outcome
if GATE3_PATH.exists():
    assert BUNDLE["bp8"]["gate3_exists"] is True
    assert BUNDLE["bp8"]["gate3_status"] == "real-run confirmed"
else:
    assert BUNDLE["bp8"]["gate3_exists"] is False
    assert BUNDLE["bp8"]["gate3_status"] == "delivered as source, not yet real-run"
_gate3_status_repr = repr(BUNDLE["bp8"]["gate3_status"])
print(f"(f) PASSED: BP8 Gate 3 code path matches Path.exists() == {GATE3_PATH.exists()} "
      f"(status: {_gate3_status_repr})")

# (g) no BP's own .docx/.xlsx/.pptx was ever opened by this notebook -- only
#     the two BP1/BP2 dashboard .html files, per the one disclosed exception
assert all(
    not str(p).endswith((".docx", ".xlsx", ".pptx"))
    for p in SOURCE_PATHS
), "This notebook must never list a BP's own .docx/.xlsx/.pptx as a source path"
_html_sources = [p for p in SOURCE_PATHS if str(p).endswith(".html")]
assert set(_html_sources) == {BP1_DASHBOARD_HTML, BP2_DASHBOARD_HTML}, (
    "Only BP1's and BP2's own dashboard HTML may be read by this notebook"
)
print("(g) PASSED: only BP1/BP2 dashboard HTML read; no BP docx/xlsx/pptx ever opened")


# (h) PDF conversion status -- SOFT check only (never asserted hard): PDF
#     conversion depends on Microsoft Word or LibreOffice being installed,
#     which this deliverable cannot assume. A "skipped" status here is a
#     WARN, not a FAIL -- every other output (HTML/DOCX/XLSX/PPTX) is
#     still required to exist via check (e) above regardless.
if PDF_RESULT["status"] == "written":
    _pdf_p = Path(PDF_RESULT["path"])
    assert _pdf_p.exists() and _pdf_p.stat().st_size > 0, (
        f"PDF_RESULT reported 'written' but {_pdf_p} is missing or empty"
    )
    print(f"(h) PASSED: PDF report written via {PDF_RESULT['method']} at {_pdf_p}")
else:
    print(f"(h) WARN (not a failure): PDF report was not generated -- {PDF_RESULT['reason']}")

print()
print("ALL STRUCTURAL INTEGRITY CHECKS PASSED.")